# Résoudre `5_energy` avec OR-Tools
Cinquième jeu de données : 11 000 modèles et 100 000 datasets. *« Looks like there's a discount on our electricity bill »* : l'énergie n'est plus la contrainte, ce sont les **données** qui manquent.

Ce notebook montre trois choses :
1. les **sources** changent de rôle : elles servent à **économiser des données** ;
2. une **astuce de modélisation** : les sources d'un même type sont interchangeables, on les agrège ;
3. un **piège numérique** : une contrainte inutile, avec d'énormes coefficients, fait annoncer au solveur un faux optimum.

**Version 3.** Ce notebook n'implémente plus rien lui-même : il appelle le package `ia_responsable` (dossier `src/`). Le récit et les expériences sont ceux de la version précédente ; le code est celui que teste `make test` et qu'utilise `make solve`. Les fonctions sont commentées dans les modules indiqués entre parenthèses.

**Durée d'exécution** : environ 15 minutes.

## 1. Charger l'instance

In [1]:
from pathlib import Path
import json, time, random
from collections import Counter
import pandas as pd

from ia_responsable import load
from ia_responsable.cli import official_score
from ia_responsable.fill import fill, give, new_pools, no_full_models
from ia_responsable.model import solve
from ia_responsable.packing import REFERENCE_ORDERS, noisy_orders, pack
from ia_responsable.pipeline import plan_value, run
from ia_responsable.repair import State, complete, repair

pd.set_option("display.width", 150)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
inst = load(ROOT / "datasets" / "5_energy.json")
WINNER = 3_786_108


def score(plan):
    """Score officiel (test_solution.py). Lève une erreur si le plan est invalide."""
    s, valid, message = official_score(ROOT, inst.text, plan)
    assert valid, message
    return s


def solver_row(info, sel):
    return dict(statut=info.status, objectif=info.objective, borne=round(info.bound, 1), écart_prouvé=round(info.gap, 1),
                secondes=round(info.seconds), pleins=len(sel.full), demi=len(sel.half), sources=len(sel.sources))


def failed_full(sel, plan):
    """Modèles choisis à pleine valeur qui n'ont pas eu leurs données libres."""
    data = {}
    for d, m in plan["dataMappings"]:
        data.setdefault(m, []).append(d)
    return [m for m in sel.full if m not in data or any(inst.copyrighted[d] for d in data[m])]


def save_if_better(plan):
    out = ROOT / "solutions" / f"{inst.name}_main.json"
    new = score(plan)
    current = official_score(ROOT, inst.text, json.loads(out.read_text()))[0] if out.exists() else 0
    if new > current:
        out.write_text(json.dumps(plan))
        print(f"{out.name} mis à jour : {current:,} -> {new:,}")
    else:
        print(f"{out.name} inchangé ({current:,}) ; ce notebook a obtenu {new:,}")


print(f"{inst.name} : {len(inst.value)} modèles, {len(inst.size)} datasets | gagnant du concours : {WINNER:,}")

5_energy : 11000 modèles, 100000 datasets | gagnant du concours : 3,786,108


## 2. Ce qui change

In [2]:
demand = {t: sum(low.get(t, 0) for low in inst.lower.values()) for t in "ntic"}
print(f"énergie de TOUS les modèles : {sum(inst.energy.values()) / inst.energy_cap:.2f} × le plafond")
print(f"datasets sous droits : {sum(inst.copyrighted.values()) / len(inst.size):.0%} | sources possibles : {len(inst.mono)}")
print("instance.energy_is_binding :", inst.energy_is_binding, "| instance.data_is_scarce :", inst.data_is_scarce)
pd.DataFrame({"offre_totale": inst.total_supply, "offre_libre": inst.free_supply, "demande_si_tout": demand,
              "demande_sur_offre": {t: round(demand[t] / inst.total_supply[t], 2) for t in "ntic"}})

énergie de TOUS les modèles : 0.92 × le plafond
datasets sous droits : 66% | sources possibles : 4554
instance.energy_is_binding : False | instance.data_is_scarce : True


,offre_totale,offre_libre,demande_si_tout,demande_sur_offre
n,126157240,42677701,127858667,1.01
t,124566959,43060085,129409873,1.04
i,124156184,43086699,134235905,1.08
c,124375133,41707631,129601799,1.04


**Lecture**
- L'énergie de **tous** les modèles réunis tient sous le plafond : **l'énergie ne compte plus**.
- La demande dépasse l'offre totale, et vaut 3 fois l'offre libre.

## 3. Les sources changent de rôle
Une source de type $t$ couvre un besoin de type $t$ **sans consommer de datasets** pour ce besoin. Son propre besoin peut être bien plus petit que celui qu'elle couvre : quand les données manquent, c'est une **économie de données**. Le prix : la cible passe à demi-valeur. Mais ici, plus de la moitié des modèles seront à demi-valeur de toute façon, faute de données libres.

## 4. Les sources agrégées (`model.py`)
Avec 4 554 sources possibles et 22 000 besoins, des variables « la source $s$ couvre le besoin $t$ de $m$ » se compteraient en **millions**. Mais une source de type $t$ peut couvrir **n'importe quel** besoin de type $t$ : seul le type compte. On **agrège** :
- $w_s$ : $s$ sert de source (à quelle cible, on le décide après) ;
- $c_{i,t}$ : le besoin $t$ de $i$ est couvert ;
- par type : $\sum_i c_{i,t} = \sum_{s \text{ de type } t} w_s$.

On passe de millions de variables à environ 50 000. L'appariement se fait après la résolution, dans `packing.pack`.

### Sources pour les besoins impossibles seulement

In [3]:
runs = {}
sel_old, info_old = solve(inst, seconds=120, possible_sources=False)
runs["sources : besoins impossibles seulement"] = solver_row(info_old, sel_old)
pd.DataFrame(runs).T

,statut,objectif,borne,écart_prouvé,secondes,pleins,demi,sources
sources : besoins impossibles seulement,FEASIBLE,3709983,3710160.8,177.8,121,4687,5221,47


**Lecture** : la **borne** de ce modèle est environ **76 000 points sous le gagnant**. Aucune solution de ce modèle ne peut donc rattraper le gagnant : **il a forcément utilisé des sources sur des besoins remplissables.** C'est la borne qui le prouve, sans même attendre l'optimum.

## 5. Le piège numérique
On lance le modèle complet en **forçant** la contrainte d'énergie, comme dans les notebooks précédents. Elle ne peut **jamais** être violée (toute l'énergie tient sous le plafond), mais elle contient 15 000 coefficients jusqu'à $10^{12}$, pour un second membre de $6 \times 10^{15}$.

In [4]:
sel_e, info_e = solve(inst, seconds=150, force_energy_constraint=True)
runs["sources partout, contrainte d'énergie forcée"] = solver_row(info_e, sel_e)
pd.DataFrame(runs).T

,statut,objectif,borne,écart_prouvé,secondes,pleins,demi,sources
sources : besoins impossibles seulement,FEASIBLE,3709983,3710160.8,177.8,121,4687,5221,47
"sources partout, contrainte d'énergie forcée",FEASIBLE,3785104,3786539.4,1435.4,155,4637,5901,238


**Ce qui s'est passé dans la version précédente.** L'énergie n'était alors pas mise à l'échelle : les coefficients valaient jusqu'à $10^{12}$. SCIP annonçait **OPTIMAL** à 3 780 183, soit **moins que le gagnant**. C'est impossible : le modèle est une relaxation du vrai problème, aucune solution réelle ne peut dépasser son optimum. Un solveur MIP calcule en virgule flottante, avec des tolérances de l'ordre de $10^{-6}$ à $10^{-9}$ ; avec de tels ordres de grandeur, les erreurs d'arrondi dans les coupes et les bornes peuvent **éliminer à tort** de bonnes solutions.

Le même piège avait d'ailleurs frappé le modèle de la partie 4 : la version précédente annonçait un « optimum prouvé » à 3 709 518, alors qu'on trouve ici une meilleure solution de ce même modèle. **Ce premier « optimum » était faux, lui aussi.**

**Ce qu'on voit maintenant.** Le package exprime l'énergie en milliards (coefficients de l'ordre de $10^3$) : même forcée, la contrainte ne produit plus de faux optimum. Mais elle reste inutile, et elle **ralentit** nettement SCIP : regardez l'écart prouvé (objectif - borne) dans le tableau.

**Remède complet** : mettre les données à l'échelle, **et** supprimer la contrainte redondante. C'est ce que fait le package par défaut (`instance.energy_is_binding`).

In [5]:
sel, info = solve(inst, seconds=300)
runs["sources partout, contrainte redondante supprimée"] = solver_row(info, sel)
pd.DataFrame(runs).T

,statut,objectif,borne,écart_prouvé,secondes,pleins,demi,sources
sources : besoins impossibles seulement,FEASIBLE,3709983,3710160.8,177.8,121,4687,5221,47
"sources partout, contrainte d'énergie forcée",FEASIBLE,3785104,3786539.4,1435.4,155,4637,5901,238
"sources partout, contrainte redondante supprimée",FEASIBLE,3786476,3786539.4,63.4,302,4638,5902,238


**Lecture** : sans la contrainte redondante, SCIP trouve une solution **au-dessus du gagnant** sur le papier, avec une borne à peine plus haute, et un écart prouvé bien plus petit. Dans la version précédente, le modèle était juste ; c'est la **résolution** qui était fausse. Le score du gagnant a servi de **témoin** : sans lui, on aurait cru à l'optimum.

## 6. Attribuer, compléter, réparer
Deux ordres de référence et 4 ordres bruités ; on complète et répare les deux meilleurs. Avec environ 5 900 modèles à demi-valeur, la réparation est limitée à **20 partenaires** d'échange (sinon elle prendrait des heures).

In [6]:
orders = dict(REFERENCE_ORDERS) | noisy_orders(inst, 4, seed=0)
packed = []
t0 = time.time()
for name, key in orders.items():
    plan, n_failed = pack(inst, sel, key)
    packed.append((plan_value(inst, plan), name, n_failed, plan))
packed.sort(key=lambda r: -r[0])
rows, best = [], (0, None, None)
for value0, name, n_failed, plan in packed[:2]:
    state = State(inst, plan)
    added = complete(state)
    n = repair(state, max_partners=20)
    s = score(state.plan())
    rows.append(dict(ordre=name, ratés=n_failed, après_attribution=value0, ajouts=added, réparations=n, score=s))
    if s > best[0]:
        best = (s, name, state.plan())
print(f"{time.time() - t0:.0f} s")
pd.DataFrame(rows).set_index("ordre")

297 s


,ratés,après_attribution,ajouts,réparations,score
ordre,,,,,
petits besoins,8,3785223,4,3,3785377
bruité 2,8,3785223,4,3,3785377


In [7]:
best_score, best_name, best_plan = best
summary = pd.DataFrame({"score": {"plan final": best_score, "sélection du solveur": info.objective, "gagnant du concours": WINNER,
                                  "borne du solveur": round(info.bound), "sans sources sur besoins remplissables": info_old.objective}})
summary["écart_au_gagnant"] = summary.score - WINNER
summary["part_du_gagnant"] = (summary.score / WINNER).map("{:.3%}".format)
display(summary)
save_if_better(best_plan)

,score,écart_au_gagnant,part_du_gagnant
plan final,3785377,-731,99.981%
sélection du solveur,3786476,368,100.010%
gagnant du concours,3786108,0,100.000%
borne du solveur,3786539,431,100.011%
sans sources sur besoins remplissables,3709983,-76125,97.989%


5_energy_main.json inchangé (3,785,420) ; ce notebook a obtenu 3,785,377


**Lecture** : la sélection du solveur dépasse le gagnant sur le papier ; tout l'écart vient de l'**attribution des datasets**. Les données libres sont utilisées à près de 100 %, et une dizaine de modèles pleins ne trouvent pas leurs données libres exactes.

## À retenir
1. **Une règle prend un sens différent selon les données** : ici, les sources économisent des données.
2. **La borne a prouvé ce qui manquait** : l'ancien modèle avait un optimum prouvé 77 000 points sous le gagnant.
3. **Agréger des objets interchangeables** : compter les sources par type, apparier après.
4. **Un solveur peut se tromper** : une contrainte inutile avec des coefficients de $10^{12}$ a produit deux faux « OPTIMAL ». Mise à l'échelle et suppression de la contrainte règlent le problème.
5. **Toujours confronter le résultat à un témoin.**